# README
- **Author**: RUEI-HSIANG, CHANG (B12303030)，Codex 協助
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
邱建富2014年彰化市長公報第1頁號次1生日欄視覺查核後，將民國49年11月29日轉為1960-11-29；與既有2009/2014 CEC彰化市長參選紀錄日期及姓名／縣市／職務相互佐證。市長是彰化市長（鄉鎮市長），不是縣市首長，因此不把這段經歷當作曾參選縣市長。只補生日與對應年齡／來源／狀態。

## What does this file take?
程式PINS逐檔固定：原始PDF、人工逐格查核JSON、既有歷史參選紀錄、81人背景表、round3主表及欄位表。原始資料與既有版本不覆寫。

## What does this file output?
- mayor_birthdate_verified_supplement_20261019.csv：1筆官方歷屆公報日期佐證。
- candidates_2026_enriched_round3_20261019.csv：81人背景表，已佐證生日由55增至56，尚缺25人。
- candidate_master_reviewed_round4_20261019.csv：最新1,583人背景狀態。
- candidate_fields_reviewed_round4_20261019.csv：最新31,660欄位清單。
所有CSV位於data/processed/。說明：data/processed/mayor_birthdate_20261019_documentation.ipynb；原始說明：data/raw/birthday_recheck_20261019_documentation.ipynb。日期有歷史官方申報佐證，不宣稱戶籍認證；2026公報待公布後再核對。


In [1]:

from pathlib import Path
import pandas as pd,hashlib,json,datetime
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
PINS={'data/raw/birthday_recheck_20261019/visual_review.json': 'd79af03d2e3abb6504d6a095170ac417f420d6898c9e268edcb67b59d29b48c1', 'data/raw/birthday_recheck_20261019/qiu_2014_bulletin.pdf': '4abfc312e471eb7522299da1cabe8399d4aa76609bea61461a797b199b12a068', 'data/processed/candidate_all_office_history.csv': '6766413aa9a810f8109adba0a1b061432f193fdfbb5fc312563bab02dce5ce70', 'data/processed/candidates_2026_enriched.csv': '291a31d7588befded91e8dce0caf7a64cc4cecc927bb4c5fbecd218f6c7281b8', 'data/processed/candidate_master_reviewed_round3_20261019.csv': 'b5656f830d321c7806d3742175f7a0761699f59f3ecd442cb9a5b3310b1c8840', 'data/processed/candidate_fields_reviewed_round3_20261019.csv': '72c245eedc0396077cf2fa5c0e821f68c41c9d566d10242b3bb2ba5b2e615144'}
for p,h in PINS.items():assert sha(ROOT/p)==h,p
r=json.loads((ROOT/'data/raw/birthday_recheck_20261019/visual_review.json').read_text())[0]
assert r['source_sha256']==sha(ROOT/r['source_file'])
assert datetime.date(49+1911,11,29).isoformat()==r['birth_date']
read=lambda p:pd.read_csv(ROOT/p,dtype=str).fillna('')
h=read('data/processed/candidate_all_office_history.csv');h=h[h.candidate_key.eq(r['candidate_key'])&h.election_year.isin(['2009','2014'])]
assert len(h)==2 and h.birth_date.eq(r['birth_date']).all() and h.office_is_county_city_mayor.eq('0').all()
e=read('data/processed/candidates_2026_enriched.csv');m=read('data/processed/candidate_master_reviewed_round3_20261019.csv');f=read('data/processed/candidate_fields_reviewed_round3_20261019.csv')
assert len(e)==81 and e.candidate_key.is_unique
mask=e.candidate_key.eq(r['candidate_key']);assert mask.sum()==1 and e.loc[mask,'birth_date_verified'].iloc[0]==''
before=int(e.birth_date_verified.eq('').sum());assert before==26
e.loc[mask,['birth_date_verified','age_2026_minus_birth_year','birth_date_source_url','birth_date_evidence_status']]=[r['birth_date'],'66',r['source_url'],'2014_official_bulletin_visually_checked_and_2009_2014_CEC_history_corroborated']
mask=m.candidate_key.eq(r['candidate_key']);assert mask.sum()==1;m.loc[mask,'birth_date_status']='官方歷屆公報逐格核對，日期已有佐證'
mask=f.candidate_key.eq(r['candidate_key'])&f.field.eq('birth_date');assert mask.sum()==1
f.loc[mask,['status','existing_value','evidence','note','next_source','updated_date']]=['官方歷屆公報逐格核對，日期已有佐證',r['birth_date'],r['source_url'],'2014公報第1頁號次1，民國49年11月29日；2009/2014 CEC日期一致。非戶籍認證。','2026公報公布後再核對','2026-10-19']
out={'data/processed/mayor_birthdate_verified_supplement_20261019.csv':pd.DataFrame([r]),'data/processed/candidates_2026_enriched_round3_20261019.csv':e,'data/processed/candidate_master_reviewed_round4_20261019.csv':m,'data/processed/candidate_fields_reviewed_round4_20261019.csv':f}
verification={}
for path,frame in out.items():
 assert path not in PINS and (ROOT/path).parent==ROOT/'data/processed'
 frame.to_csv(ROOT/path,index=False,encoding='utf-8-sig');d=read(path)
 assert list(d.columns)==list(frame.columns) and len(d)==len(frame)
 verification[path]={'rows':len(d),'sha256':sha(ROOT/path)}
assert int(e.birth_date_verified.eq('').sum())==25
old=read('data/processed/candidates_2026_enriched.csv');cols=[c for c in old if c not in ['birth_date_verified','age_2026_minus_birth_year','birth_date_source_url','birth_date_evidence_status']]
pd.testing.assert_frame_equal(e[cols],old[cols]);pd.testing.assert_frame_equal(e[~e.candidate_key.eq(r['candidate_key'])],old[~old.candidate_key.eq(r['candidate_key'])])
assert len(m)==1583 and len(f)==31660 and not f.duplicated(['candidate_key','field']).any()
(ROOT/'data/code/mayor_birthdate_20261019_verification.json').write_text(json.dumps(verification,ensure_ascii=False,indent=2))
print(json.dumps({'verified_birth_dates_before':55,'verified_birth_dates_after':56,'remaining_missing_complete_birth_dates':25,'outputs':verification},ensure_ascii=False,indent=2))


{
  "verified_birth_dates_before": 55,
  "verified_birth_dates_after": 56,
  "remaining_missing_complete_birth_dates": 25,
  "outputs": {
    "data/processed/mayor_birthdate_verified_supplement_20261019.csv": {
      "rows": 1,
      "sha256": "284471d062e9bc9474243697d1ec8e0ae0d115884626168f4b97cc95747a5472"
    },
    "data/processed/candidates_2026_enriched_round3_20261019.csv": {
      "rows": 81,
      "sha256": "e28951160f82504eb5b5d61777995ceb78a9f94bae89a724e4a9238e4c08192d"
    },
    "data/processed/candidate_master_reviewed_round4_20261019.csv": {
      "rows": 1583,
      "sha256": "8af6660bfdee31ab3f2f0189218fbc607b3d779f0c2d349575fe1d3909daa0fd"
    },
    "data/processed/candidate_fields_reviewed_round4_20261019.csv": {
      "rows": 31660,
      "sha256": "eb36a93dbe7e38e2a7dbad48741b1f7f37eea114354ca18d021951acfb30fb74"
    }
  }
}
